In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

In [2]:
# Load links
all_links = pd.read_csv("vov_coffee_links.csv")

urls = all_links["url"].tolist()

print("Total links:", len(urls))


Total links: 956


In [3]:
# Extract publish date

def extract_publish_date(soup):

    tag = soup.find("div", class_="col-md-4 mb-2")

    if tag:
        text = tag.get_text(strip=True)

        match = re.search(r"\d{2}/\d{2}/\d{4}", text)

        if match:
            return match.group()

    return None


In [4]:
# Extract price date from title

def extract_price_date(title, year):

    match = re.search(r"(\d{1,2})/(\d{1,2})", title)

    if match:

        day = match.group(1)
        month = match.group(2)

        return f"{day}/{month}/{year}"

    return None

In [5]:
# Crawl article

def crawl_article(url):

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    r = requests.get(url, headers=headers)

    soup = BeautifulSoup(r.text, "html.parser")

    # title
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else None

    # publish date
    publish_date = extract_publish_date(soup)

    # year
    year = None
    if publish_date:
        year = publish_date.split("/")[-1]

    # price date
    price_date = None
    if title and year:
        price_date = extract_price_date(title, year)

    # article text (chỉ lấy phần nội dung)
    paragraphs = soup.select("div.article-content p")

    text = " ".join([p.get_text(strip=True) for p in paragraphs])

    return {
        "url": url,
        "title": title,
        "publish_date": publish_date,
        "price_date": price_date,
        "text": text
    }

In [6]:
# Crawl all articles

dataset = []

for i, url in enumerate(urls):

    try:

        data = crawl_article(url)

        dataset.append(data)

        print(f"{i+1}/{len(urls)} done")

        time.sleep(0.8)  # tránh bị block

    except Exception as e:

        print("Error:", url)


1/956 done
2/956 done
3/956 done
4/956 done
5/956 done
6/956 done
7/956 done
8/956 done
9/956 done
10/956 done
11/956 done
12/956 done
13/956 done
14/956 done
15/956 done
16/956 done
17/956 done
18/956 done
19/956 done
20/956 done
21/956 done
22/956 done
23/956 done
24/956 done
25/956 done
26/956 done
27/956 done
28/956 done
29/956 done
30/956 done
31/956 done
32/956 done
33/956 done
34/956 done
35/956 done
36/956 done
37/956 done
38/956 done
39/956 done
40/956 done
41/956 done
42/956 done
43/956 done
44/956 done
45/956 done
46/956 done
47/956 done
48/956 done
49/956 done
50/956 done
51/956 done
52/956 done
53/956 done
54/956 done
55/956 done
56/956 done
57/956 done
58/956 done
59/956 done
60/956 done
61/956 done
62/956 done
63/956 done
64/956 done
65/956 done
66/956 done
67/956 done
68/956 done
69/956 done
70/956 done
71/956 done
72/956 done
73/956 done
74/956 done
75/956 done
76/956 done
77/956 done
78/956 done
79/956 done
80/956 done
81/956 done
82/956 done
83/956 done
84/956 done
8

In [7]:
# Create dataframe
df = pd.DataFrame(dataset)

# convert date
df["publish_date"] = pd.to_datetime(df["publish_date"], dayfirst=True)
df["price_date"] = pd.to_datetime(df["price_date"], dayfirst=True)

# sort by publish date
df = df.sort_values("publish_date")

# save
df.to_csv("vov_coffee_articles.csv", index=False)

print("Saved:", len(df), "articles")

Saved: 956 articles


In [8]:
df.head()

,url,title,publish_date,price_date,text
331,https://vov.vn/thi-truong/gia-ca-phe-hom-nay-1...,"Giá cà phê hôm nay 17/7, cao nhất 65.600 đồng/kg",2023-07-17,2023-07-17,Giá cà phê trong nướccập nhật mới nhất sáng 17...
36,https://vov.vn/thi-truong/gia-ca-phe-hom-nay-1...,Giá cà phê hôm nay 18/7: Thị trường thế giới b...,2023-07-18,2023-07-18,Giá cà phê trong nướctại các tỉnh khu vực Tây ...
686,https://vov.vn/thi-truong/gia-ca-phe-hom-nay-2...,Giá cà phê hôm nay 20/7: Giá trong nước giảm k...,2023-07-19,2023-07-20,Giá cà phê trong nước tại các tỉnh khu vực Tây...
95,https://vov.vn/thi-truong/gia-ca-phe-hom-nay-1...,Giá cà phê hôm nay 19/7: Giá trong nước và Ara...,2023-07-19,2023-07-19,Giá cà phê trong nước tại các tỉnh khu vực Tây...
225,https://vov.vn/thi-truong/gia-ca-phe-hom-nay-2...,Giá cà phê hôm nay 21/7: Giá trong nước tăng 1...,2023-07-20,2023-07-21,Giá cà phê trong nước tại các tỉnh khu vực Tây...
